In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jssunmathi/churn-modelling/Churn_Modelling.csv


In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark=SparkSession.builder.appName("Dataframe").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 06:28:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark  # when running on the local we have one master node

In [5]:
## read dataset 
## header is true so that we can remove the columns name like c:0, c:n
df = spark.read.option('header','true').csv('/kaggle/input/datasets/jssunmathi/churn-modelling/Churn_Modelling.csv')

#### df.show()  ## to show whole dataset 

In [6]:
# to check the schema similar to pandas pd.info()
df.printSchema()

root
 |-- RowNumber: string (nullable = true)
 |-- CustomerId: string (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: string (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Tenure: string (nullable = true)
 |-- Balance: string (nullable = true)
 |-- NumOfProducts: string (nullable = true)
 |-- HasCrCard: string (nullable = true)
 |-- IsActiveMember: string (nullable = true)
 |-- EstimatedSalary: string (nullable = true)
 |-- Exited: string (nullable = true)



In [7]:
type(df)  # dataframe is a data structure inside which we can perform various kinds of functions

pyspark.sql.classic.dataframe.DataFrame

In [8]:
# to get the columns name in data 
df.columns

['RowNumber',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

In [9]:
df.head(3)   # usually in pandas we get this in dataframe format but in spark it comes in the list format 

[Row(RowNumber='1', CustomerId='15634602', Surname='Hargrave', CreditScore='619', Geography='France', Gender='Female', Age='42', Tenure='2', Balance='0', NumOfProducts='1', HasCrCard='1', IsActiveMember='1', EstimatedSalary='101348.88', Exited='1'),
 Row(RowNumber='2', CustomerId='15647311', Surname='Hill', CreditScore='608', Geography='Spain', Gender='Female', Age='41', Tenure='1', Balance='83807.86', NumOfProducts='1', HasCrCard='0', IsActiveMember='1', EstimatedSalary='112542.58', Exited='0'),
 Row(RowNumber='3', CustomerId='15619304', Surname='Onio', CreditScore='502', Geography='France', Gender='Female', Age='42', Tenure='8', Balance='159660.8', NumOfProducts='3', HasCrCard='1', IsActiveMember='0', EstimatedSalary='113931.57', Exited='1')]

In [10]:
# to pickup a column 
df.select('Surname')  # written type is data frame and surname is a string

DataFrame[Surname: string]

In [11]:
df.select('Surname').show()  # to show the entire column

+---------+
|  Surname|
+---------+
| Hargrave|
|     Hill|
|     Onio|
|     Boni|
| Mitchell|
|      Chu|
| Bartlett|
|   Obinna|
|       He|
|       H?|
|   Bearce|
|  Andrews|
|      Kay|
|     Chin|
|    Scott|
|  Goforth|
|    Romeo|
|Henderson|
|  Muldrow|
|      Hao|
+---------+
only showing top 20 rows


In [12]:
type(df.select('Surname'))

pyspark.sql.classic.dataframe.DataFrame

In [13]:
## to pickup multiple columns and slicing doesn't work in pyspark and the written type is column<'Name'>

df.select(['Surname','Balance','CreditScore']).show()

+---------+---------+-----------+
|  Surname|  Balance|CreditScore|
+---------+---------+-----------+
| Hargrave|        0|        619|
|     Hill| 83807.86|        608|
|     Onio| 159660.8|        502|
|     Boni|        0|        699|
| Mitchell|125510.82|        850|
|      Chu|113755.78|        645|
| Bartlett|        0|        822|
|   Obinna|115046.74|        376|
|       He|142051.07|        501|
|       H?|134603.88|        684|
|   Bearce|102016.72|        528|
|  Andrews|        0|        497|
|      Kay|        0|        476|
|     Chin|        0|        549|
|    Scott|        0|        635|
|  Goforth|143129.41|        616|
|    Romeo|132602.88|        653|
|Henderson|        0|        549|
|  Muldrow|        0|        587|
|      Hao|        0|        726|
+---------+---------+-----------+
only showing top 20 rows


In [14]:
# to check the data types of all the data 
df.dtypes

[('RowNumber', 'string'),
 ('CustomerId', 'string'),
 ('Surname', 'string'),
 ('CreditScore', 'string'),
 ('Geography', 'string'),
 ('Gender', 'string'),
 ('Age', 'string'),
 ('Tenure', 'string'),
 ('Balance', 'string'),
 ('NumOfProducts', 'string'),
 ('HasCrCard', 'string'),
 ('IsActiveMember', 'string'),
 ('EstimatedSalary', 'string'),
 ('Exited', 'string')]

In [15]:
df.describe(['CreditScore']).show()   # for specific column

+-------+-----------------+
|summary|      CreditScore|
+-------+-----------------+
|  count|            10000|
|   mean|         650.5288|
| stddev|96.65329873613035|
|    min|              350|
|    max|              850|
+-------+-----------------+



In [16]:
df.describe().show()    # for whole dataframe

26/09/27 06:28:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|summary|         RowNumber|       CustomerId|Surname|      CreditScore|Geography|Gender|               Age|            Tenure|          Balance|     NumOfProducts|          HasCrCard|     IsActiveMember|  EstimatedSalary|             Exited|
+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|  count|             10000|            10000|  10000|            10000|    10000| 10000|             10000|             10000|            10000|             10000|              10000|              10000|            10000|              10000|
|   mean|            5000.5|

In [17]:
## adding columns in dataframe 
df = df.withColumn('Age after two years', df['Age']+2)
df.show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------------+
|RowNumber|CustomerId|  Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|Age after two years|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------------+
|        1|  15634602| Hargrave|        619|   France|Female| 42|     2|        0|            1|        1|             1|      101348.88|     1|                 44|
|        2|  15647311|     Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|                 43|
|        3|  15619304|     Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|                 44|
|        4

In [18]:
## drop the columns 
df = df.drop('Age after two years')

In [19]:
df.show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|  Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        1|  15634602| Hargrave|        619|   France|Female| 42|     2|        0|            1|        1|             1|      101348.88|     1|
|        2|  15647311|     Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        3|  15619304|     Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        4|  15701354|     Boni|        699|   France|Female| 39|     1|        0|            2|        0|             0|       93

In [20]:
## rename the columns 
df = df.withColumnRenamed('Surname','Name')
df.show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|     Name|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        1|  15634602| Hargrave|        619|   France|Female| 42|     2|        0|            1|        1|             1|      101348.88|     1|
|        2|  15647311|     Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        3|  15619304|     Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        4|  15701354|     Boni|        699|   France|Female| 39|     1|        0|            2|        0|             0|       93

In [21]:
df.show(100)

+---------+----------+-------------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|         Name|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+-------------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        1|  15634602|     Hargrave|        619|   France|Female| 42|     2|        0|            1|        1|             1|      101348.88|     1|
|        2|  15647311|         Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        3|  15619304|         Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        4|  15701354|         Boni|        699|   France|Female| 39|     1|        0|            2|      

In [22]:
df = df.na.drop()   # it will drop those rows where ever the nan or null value are present 

In [ ]:
df = df.na.drop(how='all')  # it will not drop the entire row if there is one column element is not null 
df = df.na.drop(how='any', thresh=2 )  # threshold means atleast 2 null values should be present in the row if it is it will keep that row 
df = df.na.drop(subset = ['from an specific column '])

In [ ]:
## fill the missing values 

df = df.na.fill('mising values',['specific columnn name ','one more column'])  # the null value with the 'missing value' which is passed and we can perform it on specific column


In [ ]:
from pyspark.ml.feature import Imputer 
imputer = Imputer(
    inputCols = ['age', 'Experience','Salary']
    outputCols = ['{}_imputed'.format(c) for c in ['age','Experience','Salary']]
).setStrategy('mean')  ## change the missing/null values from the mean of the entire column 

**Filter operations based on conditions **

In [25]:
# crediy score  of the people greater than equal to 700
df.filter('CreditScore>=700').show()

+---------+----------+----------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|      Name|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+----------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        5|  15737888|  Mitchell|        850|    Spain|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|
|        7|  15592531|  Bartlett|        822|   France|  Male| 50|     7|        0|            2|        1|             1|        10062.8|     0|
|       20|  15568982|       Hao|        726|   France|Female| 24|     6|        0|            2|        1|             1|       54724.03|     0|
|       21|  15577657|  McDonald|        732|   France|  Male| 41|     8|        0|            2|        1|             1|  

In [26]:
df.filter('CreditScore>=700').select(['Name','Balance','CreditScore']).show()

+----------+---------+-----------+
|      Name|  Balance|CreditScore|
+----------+---------+-----------+
|  Mitchell|125510.82|        850|
|  Bartlett|        0|        822|
|       Hao|        0|        726|
|  McDonald|        0|        732|
|       Yen|        0|        846|
|     Young|136815.64|        756|
|  Clements|        0|        722|
|   Lorenzo|  76548.6|        804|
| Armstrong|        0|        850|
|    Lavine|131394.56|        834|
|     Tyler|109421.13|        776|
|    Martin|112045.67|        829|
|    Buccho|103769.22|        776|
|   O'Brien|        0|        788|
|  Endrizzi|  75888.2|        725|
|    Hunter|   136857|        742|
|   Hammond|169831.46|        751|
|   Chibugo|123180.01|        735|
|Konovalova|133745.44|        738|
|     McKee|        0|        813|
+----------+---------+-----------+
only showing top 20 rows


In [30]:
# two different conditions 
df.filter((df['CreditScore']<=600) & 
          (df['Age']<=30)).show()

+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId|     Name|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+---------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        8|  15656148|   Obinna|        376|  Germany|Female| 29|     4|115046.74|            4|        1|             0|      119346.88|     1|
|       12|  15737173|  Andrews|        497|    Spain|  Male| 24|     3|        0|            2|        1|             0|       76390.01|     0|
|       14|  15691483|     Chin|        549|   France|Female| 25|     5|        0|            2|        0|             0|      190857.79|     0|
|       18|  15788218|Henderson|        549|    Spain|Female| 24|     9|        0|            2|        1|             1|       14

GroupBy and Aggregate Functions 

In [36]:
## groupby operations- first we have to apply the grouoby and then the aggregate functions are applied  to find the maximum salary 
df.groupBy('Name').sum().show()

+------------+
|        Name|
+------------+
|       Tyler|
|     Palermo|
|      Piccio|
|    Lazareva|
|  Kambinachi|
|       Virgo|
| Baryshnikov|
|     Wofford|
|      Lavrov|
|   Bezrukova|
|      Avdeev|
|      Clunie|
|      Duigan|
|    Sokolova|
|      Azarov|
|    Rawlings|
|         Zox|
|       Rubeo|
|      Arbour|
|Rapuluchukwu|
+------------+
only showing top 20 rows


In [40]:
## group by country and credit score 
df.groupby(['Geography','CreditScore']).avg().show()

+---------+-----------+
|Geography|CreditScore|
+---------+-----------+
|  Germany|        628|
|  Germany|        631|
|   France|        680|
|    Spain|        443|
|    Spain|        546|
|   France|        435|
|    Spain|        613|
|  Germany|        461|
|   France|        650|
|   France|        807|
|    Spain|        592|
|   France|        698|
|  Germany|        815|
|   France|        529|
|   France|        758|
|   France|        796|
|  Germany|        445|
|    Spain|        532|
|  Germany|        497|
|    Spain|        635|
+---------+-----------+
only showing top 20 rows
